# Maia Lite 0.3.0 — Auditoria e preparação do SFT científico (v1)

**Objetivo:** verificar `train-pairs.jsonl` e `raw.jsonl`, reconstruir proveniência, identificar riscos, separar **por artigo** e exportar JSONL para SFT, sem alterar os originais. **Este notebook não treina o modelo.**

**Entrada no Drive:** `MyDrive/Colab_LLMs/Maia_Lite_0_3_0_Chat/datasets/scientific_sft/` contendo `train-pairs.jsonl` e `raw.jsonl`.

**Saída:** subpasta `prepared_v1/`, com relatórios, manifesto e splits. Os arquivos de avaliação `base_420_v1/v2` **não** entram no SFT.

**Atenção:** metadados de evidência são recuperados, mas isso **não comprova automaticamente** a suficiência da evidência. O notebook identifica itens para revisão, sem declarar aprovação científica automática.

In [ ]:
from pathlib import Path
import json, hashlib, csv, random, collections, re, sys, datetime
from pprint import pprint

ROOT = Path('/content/drive/MyDrive/Colab_LLMs/Maia_Lite_0_3_0_Chat/datasets/scientific_sft')
if not ROOT.is_dir():
    try:
        from google.colab import drive
        drive.mount('/content/drive')
    except ImportError:
        print('Ambiente não-Colab: ajuste ROOT para a pasta local dos JSONL.')

PAIRS_PATH = ROOT / 'train-pairs.jsonl'
RAW_PATH = ROOT / 'raw.jsonl'
OUT = ROOT / 'prepared_v1'
for p in (PAIRS_PATH, RAW_PATH):
    if not p.is_file():
        raise FileNotFoundError(f'Arquivo ausente: {p}. Coloque os dois JSONL na pasta indicada acima.')
print('Entradas encontradas:', PAIRS_PATH, RAW_PATH, sep='\n')

In [ ]:
def sha256(path):
    h=hashlib.sha256()
    with open(path,'rb') as f:
        for block in iter(lambda:f.read(1024*1024), b''): h.update(block)
    return h.hexdigest()

def load_jsonl(path):
    rows=[]
    with open(path,encoding='utf-8') as f:
        for i,line in enumerate(f,1):
            if not line.strip(): continue
            try: rows.append(json.loads(line))
            except Exception as exc: raise ValueError(f'{path.name}, linha {i}: {exc}') from exc
    return rows

pairs=load_jsonl(PAIRS_PATH)
raw=load_jsonl(RAW_PATH)
print('Pares:',len(pairs),'Brutos:',len(raw))
print('SHA-256 pares:',sha256(PAIRS_PATH))
print('SHA-256 brutos:',sha256(RAW_PATH))
print('Campos pares:',sorted(set().union(*(r.keys() for r in pairs))))
print('Campos brutos:',sorted(set().union(*(r.keys() for r in raw))))

In [ ]:
languages={'pt','en','es'}
issues=[]
by_group=collections.defaultdict(list)
for i,r in enumerate(pairs):
    gid=r.get('groupId')
    by_group[gid].append(r)
    for k in ('id','groupId','paperId','language','question','answer'):
        if not isinstance(r.get(k),str) or not r[k].strip():
            issues.append({'kind':'missing_field','groupId':gid,'detail':f'row={i}, field={k}'})
    if r.get('language') not in languages:
        issues.append({'kind':'unexpected_language','groupId':gid,'detail':str(r.get('language'))})

ids=[r.get('id') for r in pairs]
if len(ids)!=len(set(ids)):
    issues.append({'kind':'duplicate_ids','groupId':'','detail':str(len(ids)-len(set(ids)))})
for gid,rs in by_group.items():
    langs=[r.get('language') for r in rs]
    if sorted(langs)!=sorted(languages):
        issues.append({'kind':'incomplete_translation_group','groupId':gid,'detail':str(langs)})
    if len({r.get('paperId') for r in rs})!=1:
        issues.append({'kind':'mixed_paper_in_group','groupId':gid,'detail':str([r.get('paperId') for r in rs])})
print('Grupos:',len(by_group),'Artigos:',len({r.get('paperId') for r in pairs}))
print('Idiomas:',dict(collections.Counter(r.get('language') for r in pairs)))
print('Alertas estruturais:',len(issues))
pprint(issues[:8])

In [ ]:
raw_by_id=collections.defaultdict(list)
for r in raw: raw_by_id[r.get('id')].append(r)
joined=[]
for r in pairs:
    candidates=raw_by_id.get(r.get('groupId'),[])
    same_paper=[x for x in candidates if x.get('paperId')==r.get('paperId')]
    src=same_paper[0] if len(same_paper)==1 else None
    if len(same_paper)!=1:
        issues.append({'kind':'provenance_match_count','groupId':r.get('groupId'),'detail':str(len(same_paper))})
    provenance={k:src.get(k) if src else None for k in (
        'title','doi','topic','license','sourceUrl','evidence','type','difficulty','split',
        'chunkIndex','sourceStart','sourceEnd','textSha256','pdfSha256','generator','generationFingerprint')}
    joined.append({**r,'provenance':provenance})

print('Registros unidos:',len(joined))
print('Sem associação única:',sum(x['provenance']['evidence'] is None for x in joined))
print('Licenças:',dict(collections.Counter(x['provenance']['license'] for x in joined)))
print('Splits originais:',dict(collections.Counter(x['provenance']['split'] for x in joined)))
print('Aviso: a evidência pode ser insuficiente mesmo quando não vazia.')

## Divisão por artigo e proteção contra vazamento

O arquivo `train-pairs.jsonl` deriva da partição original de treinamento. A divisão abaixo cria uma **validação interna** para monitorar o SFT; **não** cria um teste externo independente. Para avaliação científica final, reservar documentos/artigos não usados no SFT.

A divisão é feita por `paperId`, com semente fixa. O tamanho exato pode se afastar de 90/10 porque artigos possuem quantidades diferentes de questões. Todas as traduções de uma questão e todas as questões de um artigo ficam juntas.

In [ ]:
SEED=20261118
VALIDATION_FRACTION=0.10
papers=sorted({r['paperId'] for r in pairs if r.get('paperId')})
rng=random.Random(SEED)
rng.shuffle(papers)
n_val=max(1,round(len(papers)*VALIDATION_FRACTION))
val_papers=set(papers[:n_val]); train_papers=set(papers[n_val:])
assert not (val_papers & train_papers)
train=[x for x in joined if x['paperId'] in train_papers]
val=[x for x in joined if x['paperId'] in val_papers]
assert len(train)+len(val)==len(joined)
assert not ({x['groupId'] for x in train} & {x['groupId'] for x in val})
print('Train:',len(train),'registros,',len({x['groupId'] for x in train}),'grupos,',len(train_papers),'artigos')
print('Validation:',len(val),'registros,',len({x['groupId'] for x in val}),'grupos,',len(val_papers),'artigos')

## Formato de treinamento

O formato `prompt`/`completion` usa instrução fixa no idioma de cada exemplo. Para o treinamento posterior, recomendamos **loss apenas sobre a resposta** (`completion`), não sobre os tokens do prompt. Não acrescentamos tokens especiais novos ao tokenizer.

**Não usar o campo `evidence` como resposta-alvo** sem uma revisão específica. A proveniência fica em arquivo separado, e os exemplos diretos não incluem a evidência no prompt.

In [ ]:
INSTRUCTION={
 'pt':'Responda à pergunta de forma clara, objetiva e cientificamente correta. Se não souber, indique a incerteza.',
 'en':'Answer the question clearly, concisely, and scientifically accurately. If unsure, acknowledge uncertainty.',
 'es':'Responde a la pregunta de forma clara, concisa y científicamente correcta. Si no lo sabes, indica la incertidumbre.'
}
def make_sft(r):
    lang=r['language']
    return {'id':r['id'],'groupId':r['groupId'],'paperId':r['paperId'],'language':lang,
            'prompt':INSTRUCTION[lang]+'\n\n'+r['question'].strip()+'\n',
            'completion':r['answer'].strip()}

train_sft=[make_sft(x) for x in train]
val_sft=[make_sft(x) for x in val]
print('Exemplo de treinamento:')
pprint(train_sft[0])

In [ ]:
# Relatório de inspeção de qualidade; NÃO equivale a revisão científica.
flags=[]
for x in joined:
    ev=x['provenance']['evidence'] or ''
    q=x['question']; a=x['answer']
    if len(ev.strip())<80: flags.append({'id':x['id'],'reason':'short_evidence','evidence_length':len(ev)})
    if len(a.strip())<25: flags.append({'id':x['id'],'reason':'short_answer','answer_length':len(a)})
    if len(a)>3000: flags.append({'id':x['id'],'reason':'long_answer','answer_length':len(a)})
    if len(q)>1200: flags.append({'id':x['id'],'reason':'long_question','question_length':len(q)})
    if not x['provenance']['license'] or x['provenance']['license'] in ('other-oa','unknown'):
        flags.append({'id':x['id'],'reason':'license_review','license':x['provenance']['license']})
print('Alertas heurísticos (podem coexistir por item):',len(flags))
pprint(flags[:10])

In [ ]:
# Não sobrescrever resultados anteriores silenciosamente.
if OUT.exists() and any(OUT.iterdir()):
    raise FileExistsError(f'{OUT} já contém arquivos. Faça backup, renomeie a pasta ou revise antes de executar novamente.')
OUT.mkdir(parents=True,exist_ok=True)

def dump_jsonl(path,rows):
    with open(path,'w',encoding='utf-8') as f:
        for r in rows: f.write(json.dumps(r,ensure_ascii=False,sort_keys=True)+'\n')

def dump_csv(path,rows):
    if not rows: return
    keys=sorted(set().union(*(r.keys() for r in rows)))
    with open(path,'w',newline='',encoding='utf-8') as f:
        w=csv.DictWriter(f,fieldnames=keys);w.writeheader();w.writerows(rows)

dump_jsonl(OUT/'sft_train.jsonl',train_sft)
dump_jsonl(OUT/'sft_validation.jsonl',val_sft)
dump_jsonl(OUT/'provenance_all.jsonl',joined)
dump_csv(OUT/'audit_flags.csv',flags)
dump_csv(OUT/'structural_issues.csv',issues)
manifest={
 'project':'Maia Lite 0.3.0','purpose':'scientific_sft_preparation','version':'v1',
 'created_utc':datetime.datetime.now(datetime.timezone.utc).isoformat(),
 'inputs':{PAIRS_PATH.name:sha256(PAIRS_PATH),RAW_PATH.name:sha256(RAW_PATH)},
 'seed':SEED,'split_unit':'paperId','validation_fraction_target':VALIDATION_FRACTION,
 'train_records':len(train_sft),'validation_records':len(val_sft),
 'train_groups':len({x['groupId'] for x in train}),
 'validation_groups':len({x['groupId'] for x in val}),
 'train_papers':len(train_papers),'validation_papers':len(val_papers),
 'structural_issues':len(issues),'heuristic_flags':len(flags),
 'warning':'Not scientifically validated by this notebook. Human evidence and license audit remain necessary. Internal validation is not an independent test.'
}
manifest['outputs_sha256']={p.name:sha256(p) for p in sorted(OUT.iterdir()) if p.is_file()}
(OUT/'manifest.json').write_text(json.dumps(manifest,ensure_ascii=False,indent=2),encoding='utf-8')
print('Arquivos preparados:')
for p in sorted(OUT.iterdir()): print(' -',p.name,p.stat().st_size,'bytes')
pprint(manifest)

## Próximos passos

1. Revisar `structural_issues.csv`, `audit_flags.csv` e uma amostra de `provenance_all.jsonl`.
2. Confirmar as licenças dos artigos e a adequação de cada uso, especialmente ao redistribuir fragmentos.
3. Construir avaliação **externa** sem sobreposição de artigos, antes de ajustar hiperparâmetros.
4. Somente depois preparar o notebook de treinamento LoRA compatível com a arquitetura exata do checkpoint Maia Lite.

**O notebook não modifica o checkpoint nem os datasets de entrada.**